# TRM - Tiny Recursive Model Training (A100)

**Full Deep Supervision with External Datasets**

- Sudoku-Extreme from HuggingFace
- ARC-AGI from Kaggle
- Target: 87% accuracy with ~5M parameters

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q datasets wandb tqdm kaggle

## 1. Download External Datasets

In [ ]:
# Option 1: Sudoku from HuggingFace
from datasets import load_dataset

print("Loading Sudoku dataset from HuggingFace...")
# Try multiple sources
try:
    sudoku_ds = load_dataset("sadimanna/1m-sudoku", split="train[:10000]")
    print(f"Loaded {len(sudoku_ds)} examples from sadimanna/1m-sudoku")
except:
    try:
        sudoku_ds = load_dataset("Ritvik19/Sudoku-Dataset", split="train[:10000]")
        print(f"Loaded {len(sudoku_ds)} examples from Ritvik19/Sudoku-Dataset")
    except Exception as e:
        print(f"Could not load from HuggingFace: {e}")
        sudoku_ds = None

if sudoku_ds:
    print("Sample:", sudoku_ds[0])

In [ ]:
# Option 2: Download from Kaggle (requires API key)
# Uncomment and add your Kaggle credentials if HuggingFace fails

# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'
# !kaggle datasets download -d bryanpark/sudoku -p ./data --unzip
# !kaggle datasets download -d mexwell/maze-dataset -p ./data --unzip

In [ ]:
# Process Sudoku dataset into training format
import numpy as np

def parse_sudoku_string(puzzle_str, solution_str):
    """Convert string format to numpy arrays"""
    puzzle = np.array([int(c) for c in str(puzzle_str)], dtype=np.int64)
    solution = np.array([int(c) for c in str(solution_str)], dtype=np.int64)
    return puzzle, solution

# Prepare data
puzzles = []
solutions = []

if sudoku_ds:
    for ex in sudoku_ds:
        try:
            # Handle different column names
            if 'puzzle' in ex:
                p, s = parse_sudoku_string(ex['puzzle'], ex['solution'])
            elif 'quizzes' in ex:
                p, s = parse_sudoku_string(ex['quizzes'], ex['solutions'])
            else:
                continue
            if len(p) == 81 and len(s) == 81:
                puzzles.append(p)
                solutions.append(s)
        except:
            continue

print(f"Processed {len(puzzles)} valid puzzles")

## 2. Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

class SwiGLU(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        hidden = ((int(2/3 * 4 * in_dim) + 63) // 64) * 64
        self.w1 = nn.Linear(in_dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, in_dim, bias=False)
        self.w3 = nn.Linear(in_dim, hidden, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TinyBlock(nn.Module):
    def __init__(self, dim, n_heads=8):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True, bias=False)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLU(dim)
    
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h)[0]
        x = x + self.ffn(self.norm2(x))
        return x

class TinyNet(nn.Module):
    def __init__(self, dim=512, n_layers=2, n_heads=8):
        super().__init__()
        self.blocks = nn.ModuleList([TinyBlock(dim, n_heads) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

class TRMModel(nn.Module):
    """TRM with deep recursion - Algorithm 3"""
    def __init__(self, dim=512, n_layers=2, n_heads=8, n_latent=6, T_cycles=3, vocab_size=10):
        super().__init__()
        self.dim = dim
        self.n_latent = n_latent
        self.T_cycles = T_cycles
        
        self.net = TinyNet(dim, n_layers, n_heads)
        self.z_proj = nn.Linear(3 * dim, dim, bias=False)
        self.y_proj = nn.Linear(2 * dim, dim, bias=False)
        self.output_head = nn.Linear(dim, vocab_size, bias=False)
        self.q_head = nn.Linear(dim, 1, bias=False)
    
    def latent_recursion(self, x, y, z):
        """n latent z-updates + 1 y-refine"""
        for _ in range(self.n_latent):
            z = self.net(self.z_proj(torch.cat([x, y, z], dim=-1)))
        y = self.net(self.y_proj(torch.cat([y, z], dim=-1)))
        return y, z
    
    def forward(self, x, y, z):
        # T-1 no-grad warmups
        with torch.no_grad():
            for _ in range(self.T_cycles - 1):
                y, z = self.latent_recursion(x, y, z)
        # Final cycle with gradients
        y, z = self.latent_recursion(x, y, z)
        y_hat = self.output_head(y)
        q_hat = self.q_head(y.mean(dim=1))
        return (y, z), y_hat, q_hat
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Test model
model = TRMModel(dim=512, n_layers=2, n_latent=6, T_cycles=3)
print(f"TRM Parameters: {model.count_parameters():,}")

## 3. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SudokuDataset(Dataset):
    def __init__(self, puzzles, solutions):
        self.puzzles = puzzles
        self.solutions = solutions
    
    def __len__(self):
        return len(self.puzzles)
    
    def __getitem__(self, idx):
        puzzle = torch.tensor(self.puzzles[idx], dtype=torch.long)
        solution = torch.tensor(self.solutions[idx], dtype=torch.long)
        # Random init for blanks (1-9)
        y_init = puzzle.clone()
        blank_mask = puzzle == 0
        y_init[blank_mask] = torch.randint(1, 10, (blank_mask.sum(),))
        return puzzle, y_init, solution

# Split train/val
split = int(0.9 * len(puzzles))
train_ds = SudokuDataset(puzzles[:split], solutions[:split])
val_ds = SudokuDataset(puzzles[split:], solutions[split:])

# A100 can handle larger batches
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## 4. Training with Full Deep Supervision (N_sup=16)

In [ ]:
import time
from tqdm.auto import tqdm

# A100 Optimized Config
CONFIG = {
    'dim': 512,
    'n_layers': 2,
    'n_latent': 6,        # Latent z-updates per cycle
    'T_cycles': 3,        # Recursion cycles (42+ effective layers)
    'N_sup': 16,          # FULL deep supervision
    'batch_size': 128,    # A100 can handle this
    'lr': 1e-4,
    'max_iters': 50000,
    'halt_threshold': 0.5,
    'ema_beta': 0.999,
}

device = 'cuda'
model = TRMModel(
    dim=CONFIG['dim'],
    n_layers=CONFIG['n_layers'],
    n_latent=CONFIG['n_latent'],
    T_cycles=CONFIG['T_cycles']
).to(device)

embedding = nn.Embedding(20, CONFIG['dim']).to(device)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(embedding.parameters()),
    lr=CONFIG['lr'],
    weight_decay=0.01,
    betas=(0.9, 0.95)
)

# Mixed precision for A100
scaler = torch.amp.GradScaler('cuda')

# EMA for stable generalization
ema_params = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
for n, p in embedding.named_parameters():
    ema_params[f'emb_{n}'] = p.data.clone()

def update_ema():
    beta = CONFIG['ema_beta']
    for n, p in model.named_parameters():
        if n in ema_params:
            ema_params[n] = beta * ema_params[n] + (1-beta) * p.data
    for n, p in embedding.named_parameters():
        key = f'emb_{n}'
        if key in ema_params:
            ema_params[key] = beta * ema_params[key] + (1-beta) * p.data

print(f"Model: {model.count_parameters():,} params")
print(f"Config: N_sup={CONFIG['N_sup']}, T={CONFIG['T_cycles']}, n={CONFIG['n_latent']}")
print(f"Effective depth: {CONFIG['T_cycles'] * (CONFIG['n_latent'] + 1) * 2} layers")

In [ ]:
# Training loop with FULL deep supervision
train_iter = iter(train_loader)
start = time.time()
best_val_acc = 0

for step in tqdm(range(CONFIG['max_iters']), desc="Training"):
    model.train()
    
    try:
        puzzle, y_init_tokens, solution = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        puzzle, y_init_tokens, solution = next(train_iter)
    
    puzzle = puzzle.to(device)
    y_init_tokens = y_init_tokens.to(device)
    solution = solution.to(device)
    
    # Embed tokens
    x = embedding(puzzle)
    y = embedding(y_init_tokens)
    z = torch.zeros_like(x)
    
    total_loss = 0
    steps_taken = 0
    
    # FULL Deep Supervision Loop (N_sup=16)
    for sup_step in range(CONFIG['N_sup']):
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            (y_new, z_new), y_hat, q_hat = model(x, y, z)
            
            # CE loss
            ce_loss = F.cross_entropy(y_hat.view(-1, 10), solution.view(-1), ignore_index=0)
            
            # ACT halt target
            with torch.no_grad():
                preds = y_hat.argmax(-1)
                acc = (preds == solution).float().mean(dim=-1)
                halt_target = (acc > 0.95).float().unsqueeze(-1)
            
            act_loss = F.binary_cross_entropy_with_logits(q_hat, halt_target)
            loss = ce_loss + 0.5 * act_loss
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        update_ema()
        
        total_loss += loss.item()
        steps_taken += 1
        
        # Early halt if confident
        halt_prob = torch.sigmoid(q_hat).mean().item()
        if halt_prob > CONFIG['halt_threshold']:
            break
        
        # Detach and continue
        y = y_new.detach()
        z = z_new.detach()
    
    # Logging
    if step % 100 == 0:
        with torch.no_grad():
            train_acc = (y_hat.argmax(-1) == solution).float().mean().item()
        elapsed = time.time() - start
        print(f"Step {step} | Loss: {total_loss/steps_taken:.4f} | Acc: {train_acc:.3f} | "
              f"SupSteps: {steps_taken} | Time: {elapsed:.0f}s")
    
    # Validation
    if step % 1000 == 0 and step > 0:
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for puzzle, y_init, solution in val_loader:
                puzzle, y_init, solution = puzzle.to(device), y_init.to(device), solution.to(device)
                x = embedding(puzzle)
                y = embedding(y_init)
                z = torch.zeros_like(x)
                
                for _ in range(CONFIG['N_sup']):
                    (y, z), y_hat, q_hat = model(x, y, z)
                    if torch.sigmoid(q_hat).mean() > 0.5:
                        break
                
                val_correct += (y_hat.argmax(-1) == solution).float().mean().item() * puzzle.size(0)
                val_total += puzzle.size(0)
        
        val_acc = val_correct / val_total
        print(f"  VAL Acc: {val_acc:.3f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'model': model.state_dict(),
                'embedding': embedding.state_dict(),
                'ema': ema_params,
                'step': step,
                'val_acc': val_acc
            }, 'trm_best.pt')
            print(f"  Saved best model (acc: {val_acc:.3f})")

# Save final
torch.save({
    'model': model.state_dict(),
    'embedding': embedding.state_dict(),
    'ema': ema_params,
    'step': CONFIG['max_iters']
}, 'trm_final.pt')
print(f"\nTraining complete! Best val acc: {best_val_acc:.3f}")

## 5. Final Evaluation with EMA

In [ ]:
# Load best checkpoint with EMA weights
ckpt = torch.load('trm_best.pt')
model.load_state_dict(ckpt['model'])

# Apply EMA weights
for n, p in model.named_parameters():
    if n in ckpt['ema']:
        p.data = ckpt['ema'][n]

model.eval()

# Full evaluation
correct_cells = 0
total_cells = 0
correct_grids = 0
total_grids = 0

with torch.no_grad():
    for puzzle, y_init, solution in tqdm(val_loader, desc="Evaluating"):
        puzzle, y_init, solution = puzzle.to(device), y_init.to(device), solution.to(device)
        x = embedding(puzzle)
        y = embedding(y_init)
        z = torch.zeros_like(x)
        
        for _ in range(CONFIG['N_sup']):
            (y, z), y_hat, q_hat = model(x, y, z)
            if torch.sigmoid(q_hat).mean() > 0.5:
                break
        
        preds = y_hat.argmax(-1)
        correct_cells += (preds == solution).sum().item()
        total_cells += solution.numel()
        correct_grids += (preds == solution).all(dim=-1).sum().item()
        total_grids += puzzle.size(0)

print(f"\n=== Final Results ===")
print(f"Cell Accuracy: {100*correct_cells/total_cells:.2f}%")
print(f"Grid Accuracy: {100*correct_grids/total_grids:.2f}%")

In [ ]:
# Download checkpoints
from google.colab import files
files.download('trm_best.pt')
files.download('trm_final.pt')